# A/B 공통 데이터 준비
KILT Natural Questions(단일 자료)와 HotpotQA(여러 자료)의 질문·정답·근거 위치를 분리한다.
두 방법은 동일한 원문 저장소를 사용한다. 공식 KILT Wikipedia 원본은 약 35GiB다.

이미 준비된 manifest가 있으면 해시를 검증한다. `REBUILD=True`일 때만 동일 규칙으로 다시 추출한다.
데이터 준비 시간은 온라인 질의 시간과 구별한다.

### 1. 환경과 고정 설정

In [ ]:
from pathlib import Path
import sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'config/baselines.json').exists():
    ROOT = ROOT.parent
assert (ROOT / 'config/baselines.json').exists(), '프로젝트 루트에서 열어주세요.'
sys.path.insert(0, str(ROOT))
from rag_baselines.data import read_json, read_jsonl, file_sha256, write_json, write_jsonl
import pandas as pd
pd.set_option('display.max_colwidth', 90)
CONFIG = read_json(ROOT / 'config/baselines.json')
print('Workspace:', ROOT)

In [ ]:
REBUILD = False
from rag_baselines.data import (download_wikipedia, prepare_question_candidates,
                                extract_articles, build_common_corpora)
display(pd.DataFrame([
    {'component': 'BERGEN', 'revision': CONFIG['bergen_revision']},
    {'component': CONFIG['retriever']['repo'], 'revision': CONFIG['retriever']['revision']},
    {'component': CONFIG['reranker']['repo'], 'revision': CONFIG['reranker']['revision']},
    {'component': CONFIG['qa']['repo'], 'revision': CONFIG['qa']['revision']},
]))

### 2. 공식 원본과 문서 준비
출처: [KILT](https://github.com/facebookresearch/KILT#kilt-knowledge-source),
[KILT 질문](https://huggingface.co/datasets/facebook/kilt_tasks).
100 **단어** 단위로 나누고 제목을 붙이는 BERGEN KILT100w 규칙을 따른다.
BULLET 문단은 제외하며 원래 문단·문자 위치는 따로 보존한다.

In [ ]:
started = time.perf_counter()
manifest_path = ROOT / 'data/prepared/manifest.json'
if REBUILD or not manifest_path.exists():
    original = download_wikipedia(ROOT)
    candidates = prepare_question_candidates(ROOT)
    articles, source_meta = extract_articles(ROOT, candidates)
    manifest = build_common_corpora(ROOT, candidates, articles, source_meta)
    manifest['prepare_seconds'] = time.perf_counter() - started
    write_json(manifest_path, manifest)
    reused = False
else:
    manifest = read_json(manifest_path)
    assert manifest['config'] == CONFIG, '설정 변경 시 별도 데이터 버전을 준비하세요.'
    reused = True
print('prepared cache reused:', reused)
print('original extraction/preparation seconds:', round(manifest.get('prepare_seconds', 0), 2))
print('source articles:', manifest['source']['source_articles'])
print('source SHA256:', manifest['source']['source_sha256'])
display(pd.DataFrame(manifest['question_counts']).T)

### 3. 공통 corpus와 근거 위치 검증
정답 근거를 강제로 포함한 **축소 corpus 실험**이다. KILT 전체 corpus 결과가 아니다.
배경은 전체 원본에서 균일한 문서 15,000개를 뽑고, 보관된 문서의 passage에서 표본 추출한다.
전체 Wikipedia passage에 대한 균일 표본이라고 해석하지 않는다.

In [ ]:
corpus_sets, rows = [], []
for size, entry in sorted(manifest['corpora'].items(), key=lambda item: int(item[0])):
    path = ROOT / entry['path']
    assert file_sha256(path) == entry['sha256']
    passages = read_jsonl(path)
    ids = {p['id'] for p in passages}
    assert len(ids) == entry['count']
    corpus_sets.append(ids)
    rows.append({'passages': len(ids), 'bytes': path.stat().st_size, 'sha256': entry['sha256'][:16]})
assert corpus_sets[0] < corpus_sets[1] < corpus_sets[2]
for split in ('dev', 'eval'):
    for kind in ('questions', 'labels'):
        path = ROOT / f'data/prepared/{kind}_{split}.jsonl'
        assert file_sha256(path) == manifest[f'{kind}_{split}_sha256']
    questions = read_jsonl(ROOT / f'data/prepared/questions_{split}.jsonl')
    labels = read_jsonl(ROOT / f'data/prepared/labels_{split}.jsonl')
    assert {q['id'] for q in questions} == {label['id'] for label in labels}
    assert all('answers' not in q and 'evidence_groups' not in q for q in questions)
    assert all(set(group) <= corpus_sets[0] for label in labels for group in label['evidence_groups'])
    assert all(len({pid.split(':')[0] for pid in group}) >= 2 for label in labels if label['task']=='hotpotqa' for group in label['evidence_groups'])
display(pd.DataFrame(rows))
print('All hashes, nested corpus IDs, label separation and joint multi-document groups passed.')
print('excluded questions:', len(manifest['exclusions']))

### 4. 저장된 준비 비용
시간·디스크·처리 토큰은 각각 별도 단위다. API를 호출하지 않은 단계의 API 비용은 측정 대상이 아니다.
최초 다운로드의 활성 시간은 기록하지 않았으므로 추정해서 채우지 않는다.

In [ ]:
download = read_json(ROOT / 'data/raw/kilt_wikipedia/download.json')
qa_files = sorted((ROOT / 'data/raw/kilt_tasks').glob('*/*.parquet'))
preparation = {
    'source_bytes': CONFIG['wikipedia']['bytes'],
    'qa_files': {str(p.relative_to(ROOT)): {'bytes': p.stat().st_size, 'sha256': file_sha256(p)} for p in qa_files},
    'prepared_corpus_bytes': sum((ROOT / entry['path']).stat().st_size for entry in manifest['corpora'].values()),
    'extraction_and_chunking_seconds': manifest.get('prepare_seconds'),
    'download_active_seconds': download.get('download_seconds'),
    'download_time_note': 'initial download active time unavailable' if 'download_seconds' not in download else None,
    'current_validation_seconds': time.perf_counter() - started,
    'api_usage': None, 'api_cost_usd': None,
}
write_json(ROOT / 'data/prepared/preparation_metrics.json', preparation)
display(pd.DataFrame([{k: v for k,v in preparation.items() if k != 'qa_files'}]))

다음은 `02_measure_retrieval.ipynb`. 개발셋으로 동작을 확인하고, 설정을 고정한 뒤 평가셋을 사용한다.